<a href="https://colab.research.google.com/github/shinmyatnoezin/HousePredictionProject/blob/main/HousePrediction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 1. Import Libraries

In [ ]:
# =========================
# 1. IMPORT LIBRARIES
# =========================
import pandas as pd
import numpy as np
import glob
import os
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import (
    mean_squared_error,
    r2_score,
    mean_absolute_error,
    median_absolute_error,
    explained_variance_score,
    max_error
)

sns.set_style("whitegrid")

# 2. Load HM Land Registry Data

In [ ]:
# =========================
# 2. LOAD HM LAND REGISTRY DATA
# =========================

files = glob.glob("data/*.csv")

if len(files) == 0:
    raise FileNotFoundError("No CSV files found in the 'data' folder.")

df_list = []
for file in files:
    temp = pd.read_csv(file, header=None, low_memory=False)
    df_list.append(temp)

df = pd.concat(df_list, ignore_index=True)

# Assign HM Land Registry column names
df.columns = [
    "transaction_id", "price", "date", "postcode",
    "property_type", "new_build", "tenure",
    "paon", "saon", "street", "locality",
    "town_city", "district", "county",
    "ppd_category", "record_status"
]

print("Land Registry data loaded:", df.shape)
print(df.head())

# 3. Basic Cleaning

In [ ]:
# =========================
# 3. BASIC CLEANING
# =========================
df = df[[
    "price", "date", "postcode", "property_type",
    "new_build", "tenure", "town_city", "district", "county"
]].copy()

# Clean strings
for col in ["postcode", "property_type", "new_build", "tenure", "town_city", "district", "county"]:
    df[col] = df[col].astype(str).str.strip()

# Convert date and price
df["date"] = pd.to_datetime(df["date"], errors="coerce")
df["price"] = pd.to_numeric(df["price"], errors="coerce")

# Remove missing important values
df = df.dropna(subset=["price", "postcode", "date"])

# Remove impossible / suspicious prices if needed
df = df[df["price"] > 0]

print("After basic cleaning:", df.shape)

# 4. Load UK Postcodes Data

In [ ]:
# =========================
# 4. LOAD UK POSTCODES DATA
# =========================
postcode_file = "ukpostcodes.csv"

postcode_df = pd.read_csv(postcode_file, low_memory=False)

print("Postcode dataset columns:")
print(postcode_df.columns.tolist())

needed_cols = ["postcode", "latitude", "longitude"]
for col in needed_cols:
    if col not in postcode_df.columns:
        raise KeyError(f"Column '{col}' not found in postcode dataset.")

postcode_df = postcode_df[["postcode", "latitude", "longitude"]].copy()
postcode_df["postcode"] = postcode_df["postcode"].astype(str).str.strip()
postcode_df["latitude"] = pd.to_numeric(postcode_df["latitude"], errors="coerce")
postcode_df["longitude"] = pd.to_numeric(postcode_df["longitude"], errors="coerce")

postcode_df = postcode_df.dropna(subset=["postcode", "latitude", "longitude"])
postcode_df = postcode_df.drop_duplicates(subset=["postcode"])

print("Postcode data cleaned:", postcode_df.shape)

# 5. Merge Land Registry + Postcodes

In [ ]:
# =========================
# 5. MERGE LAND REGISTRY + POSTCODES
# =========================
df = df.merge(postcode_df, on="postcode", how="left")

print("After merge:", df.shape)
print("Missing latitude:", df["latitude"].isna().sum())
print("Missing longitude:", df["longitude"].isna().sum())

# Drop rows where coordinates were not found
df = df.dropna(subset=["latitude", "longitude"]).copy()

print("After dropping missing coordinates:", df.shape)

# 6. Filter To A Specific Area

In [ ]:
# =========================
# 6. FILTER TO A SPECIFIC AREA
# =========================
df = df[df["district"].str.contains("BRISTOL", case=False, na=False)].copy()
print("After Bristol filter:", df.shape)


# 7. Create Extra Location-Relate Features

**7.1 Year And Month From Sale Date**

In [ ]:
# =========================
# 7. CREATE EXTRA LOCATION-RELATED FEATURES
# =========================

# 7.1 Year and month from sale date
df["year"] = df["date"].dt.year
df["month"] = df["date"].dt.month

**7.2 Bristol Approximate Center Reference Point**

In [ ]:
# 7.2 Bristol approximate center reference point
reference_lat = 51.4545   # Bristol latitude
reference_lon = -2.5879   # Bristol longitude

# Euclidean distance in coordinate space
df["distance_to_reference"] = np.sqrt(
    (df["latitude"] - reference_lat) ** 2 +
    (df["longitude"] - reference_lon) ** 2
)


**7.3 Interaction Feature**

In [ ]:
# 7.3 Interaction feature
df["lat_long_interaction"] = df["latitude"] * df["longitude"]

**7.4 Coordinate Bins (Spatial Grouping)**

In [ ]:
# 7.4 Coordinate bins (spatial grouping)
df["lat_bin"] = pd.cut(df["latitude"], bins=10, labels=False)
df["lon_bin"] = pd.cut(df["longitude"], bins=10, labels=False)

**7.5 KMeans Location Clusters**

In [ ]:
# 7.5 KMeans location clusters
kmeans = KMeans(n_clusters=8, random_state=42, n_init=10)
df["location_cluster"] = kmeans.fit_predict(df[["latitude", "longitude"]])

**7.6 Cluster Average Price**

In [ ]:
# 7.6 Cluster average price
cluster_avg = df.groupby("location_cluster")["price"].mean()
df["cluster_avg_price"] = df["location_cluster"].map(cluster_avg)

**7.7 District Average Price**

In [ ]:
# 7.7 District average price
district_avg = df.groupby("district")["price"].mean()
df["district_avg_price"] = df["district"].map(district_avg)

**7.8 Town Average Price**

In [ ]:
# 7.8 Town average price
town_avg = df.groupby("town_city")["price"].mean()
df["town_avg_price"] = df["town_city"].map(town_avg)

print("Feature engineering complete.")
print(df.head())

# 8. Handle Outliers

In [ ]:
# =========================
# 8. HANDLE OUTLIERS
# =========================
# Remove extreme top 1% prices to make model more stable
upper_limit = df["price"].quantile(0.99)
df = df[df["price"] <= upper_limit].copy()

print("After removing top 1% price outliers:", df.shape)

# 9. EDA Analysis

**9.1 Dataset overview**

In [ ]:
# =========================
# 9. EDA ANALYSIS
# =========================

# 9.1 Dataset overview
print("\nData types:\n", df.dtypes)
print("\nSummary statistics:\n", df.describe(include="all"))

**9.2 Missing Values**

In [ ]:
# 9.2 Missing values
print("\nMissing values:\n", df.isnull().sum())

**9.3 Distribution Of Price**

In [ ]:
# 9.3 Distribution of price
plt.figure(figsize=(10, 5))
sns.histplot(df["price"], bins=50, kde=True)
plt.title("Distribution of House Prices")
plt.xlabel("Price")
plt.ylabel("Count")
plt.tight_layout()
plt.show()


**9.4 Log Price Distribution**

In [ ]:
# 9.4 Log price distribution
df["log_price"] = np.log1p(df["price"])

plt.figure(figsize=(10, 5))
sns.histplot(df["log_price"], bins=50, kde=True)
plt.title("Distribution of Log House Prices")
plt.xlabel("Log Price")
plt.ylabel("Count")
plt.tight_layout()
plt.show()

**9.5 Scatter Plot Of Longitude vs Latitude Colored By Price**

In [ ]:
# 9.5 Scatter plot of longitude vs latitude colored by price
sample_df = df.sample(min(5000, len(df)), random_state=42)

plt.figure(figsize=(10, 6))
scatter = plt.scatter(
    sample_df["longitude"],
    sample_df["latitude"],
    c=sample_df["price"],
    s=10,
    alpha=0.7
)
plt.colorbar(scatter, label="Price")
plt.title("House Prices by Geographic Location")
plt.xlabel("Longitude")
plt.ylabel("Latitude")
plt.tight_layout()
plt.show()

**9.6 Boxplot By Property Type**

In [ ]:
# 9.6 Boxplot by property type
plt.figure(figsize=(8, 5))
sns.boxplot(x="property_type", y="price", data=df)
plt.title("House Price by Property Type")
plt.xlabel("Property Type")
plt.ylabel("Price")
plt.tight_layout()
plt.show()

**9.7 Transactions Per Year**

In [ ]:
# 9.7 Transactions per year
plt.figure(figsize=(8, 5))
sns.countplot(x="year", data=df, order=sorted(df["year"].dropna().unique()))
plt.title("Number of Transactions per Year")
plt.xlabel("Year")
plt.ylabel("Count")
plt.tight_layout()
plt.show()


**9.8 Average Price By Year**

In [ ]:
# 9.8 Average price by year
avg_price_year = df.groupby("year")["price"].mean().reset_index()

plt.figure(figsize=(8, 5))
plt.plot(avg_price_year["year"], avg_price_year["price"], marker="o")
plt.title("Average House Price by Year")
plt.xlabel("Year")
plt.ylabel("Average Price")
plt.tight_layout()
plt.show()

**9.9 Correlation heatmap**

In [ ]:
# 9.9 Correlation heatmap (numeric only)
numeric_cols = df.select_dtypes(include=[np.number]).columns
corr = df[numeric_cols].corr()

plt.figure(figsize=(12, 8))
sns.heatmap(corr, cmap="coolwarm", center=0)
plt.title("Correlation Heatmap")
plt.tight_layout()
plt.show()


**9.10 Cluster Visualization**

In [ ]:
# 9.10 Cluster visualization
plt.figure(figsize=(10, 6))
plt.scatter(
    sample_df["longitude"],
    sample_df["latitude"],
    c=sample_df["location_cluster"],
    s=10,
    alpha=0.7
)
plt.title("Location Clusters")
plt.xlabel("Longitude")
plt.ylabel("Latitude")
plt.tight_layout()
plt.show()

# 10. Prepare Data For Model

In [ ]:
# =========================
# 10. PREPARE DATA FOR MODEL
# =========================

# Choose features
feature_cols = [
    "latitude",
    "longitude",
    "distance_to_reference",
    "lat_long_interaction",
    "lat_bin",
    "lon_bin",
    "location_cluster",
    "cluster_avg_price",
    "district_avg_price",
    "town_avg_price",
    "year",
    "month"
]

# Add categorical columns that are relevant
categorical_cols = ["property_type", "new_build", "tenure"]

model_df = df[feature_cols + categorical_cols + ["price", "log_price"]].copy()

# One-hot encode categorical columns
model_df = pd.get_dummies(model_df, columns=categorical_cols, drop_first=True)

# Final X and y
X = model_df.drop(columns=["price", "log_price"])
y = model_df["log_price"]   # predicting log price often works better

print("Final model data shape:", model_df.shape)
print("Feature matrix shape:", X.shape)


# 11. Train / Test Split

In [ ]:
# =========================
# 11. TRAIN / TEST SPLIT
# =========================
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# 12. Train Models

**12.1 Linear Regression**

In [ ]:
# =========================
# 12. TRAIN MODELS
# =========================

# 12.1 Linear Regression
lr_model = LinearRegression()
lr_model.fit(X_train, y_train)
lr_pred_log = lr_model.predict(X_test)

**12.2 Random Forest**

In [ ]:
# 12.2 Random Forest
rf_model = RandomForestRegressor(
    n_estimators=200,
    max_depth=20,
    min_samples_split=10,
    min_samples_leaf=4,
    random_state=42,
    n_jobs=-1
)
rf_model.fit(X_train, y_train)
rf_pred_log = rf_model.predict(X_test)

# Convert predictions back to original price scale
y_test_actual = np.expm1(y_test)
lr_pred = np.expm1(lr_pred_log)
rf_pred = np.expm1(rf_pred_log)

# 13. Evaluation Function

In [ ]:
# =========================
# 13. EVALUATION FUNCTION
# =========================
def evaluate_model(y_true, y_pred, model_name="Model"):
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2 = r2_score(y_true, y_pred)
    mae = mean_absolute_error(y_true, y_pred)
    medae = median_absolute_error(y_true, y_pred)
    evs = explained_variance_score(y_true, y_pred)
    maxerr = max_error(y_true, y_pred)

    # MAPE with protection against divide-by-zero
    y_true_safe = np.where(y_true == 0, 1, y_true)
    mape = np.mean(np.abs((y_true - y_pred) / y_true_safe)) * 100

    # SMAPE
    smape = 100 * np.mean(
        2 * np.abs(y_pred - y_true) / (np.abs(y_true) + np.abs(y_pred) + 1e-10)
    )

    print(f"\n===== {model_name} Evaluation =====")
    print(f"RMSE: {rmse:,.2f}")
    print(f"R²: {r2:.4f}")
    print(f"MAE: {mae:,.2f}")
    print(f"Median Absolute Error: {medae:,.2f}")
    print(f"MAPE: {mape:.2f}%")
    print(f"SMAPE: {smape:.2f}%")
    print(f"Explained Variance Score: {evs:.4f}")
    print(f"Max Error: {maxerr:,.2f}")

    return {
        "RMSE": rmse,
        "R2": r2,
        "MAE": mae,
        "MedianAE": medae,
        "MAPE": mape,
        "SMAPE": smape,
        "ExplainedVariance": evs,
        "MaxError": maxerr
    }

# Evaluate both models
lr_results = evaluate_model(y_test_actual, lr_pred, "Linear Regression")
rf_results = evaluate_model(y_test_actual, rf_pred, "Random Forest")

# 14. Actual VS Predicted Plots

In [ ]:
# =========================
# 14. ACTUAL VS PREDICTED PLOTS
# =========================
plt.figure(figsize=(7, 6))
plt.scatter(y_test_actual, lr_pred, alpha=0.5)
plt.xlabel("Actual Price")
plt.ylabel("Predicted Price")
plt.title("Linear Regression: Actual vs Predicted")
plt.tight_layout()
plt.show()

plt.figure(figsize=(7, 6))
plt.scatter(y_test_actual, rf_pred, alpha=0.5)
plt.xlabel("Actual Price")
plt.ylabel("Predicted Price")
plt.title("Random Forest: Actual vs Predicted")
plt.tight_layout()
plt.show()


# 15. Residual Plots

In [ ]:
# =========================
# 15. RESIDUAL PLOTS
# =========================
lr_residuals = y_test_actual - lr_pred
rf_residuals = y_test_actual - rf_pred

plt.figure(figsize=(7, 6))
plt.scatter(lr_pred, lr_residuals, alpha=0.5)
plt.axhline(0, linestyle="--")
plt.xlabel("Predicted Price")
plt.ylabel("Residuals")
plt.title("Linear Regression Residual Plot")
plt.tight_layout()
plt.show()

plt.figure(figsize=(7, 6))
plt.scatter(rf_pred, rf_residuals, alpha=0.5)
plt.axhline(0, linestyle="--")
plt.xlabel("Predicted Price")
plt.ylabel("Residuals")
plt.title("Random Forest Residual Plot")
plt.tight_layout()
plt.show()

# 16. Feature Importance For Random Forest

In [ ]:
# =========================
# 16. FEATURE IMPORTANCE FOR RANDOM FOREST
# =========================
importances = pd.Series(rf_model.feature_importances_, index=X.columns).sort_values(ascending=False)

plt.figure(figsize=(10, 8))
importances.head(15).sort_values().plot(kind="barh")
plt.title("Top 15 Most Important Features (Random Forest)")
plt.xlabel("Importance")
plt.tight_layout()
plt.show()

print("\nTop 15 important features:")
print(importances.head(15))

# 17. Save Final Dataset

In [ ]:
# =========================
# 17. SAVE FINAL DATASET
# =========================
df.to_csv("final_house_price_dataset_with_features.csv", index=False)
print("\nSaved: final_house_price_dataset_with_features.csv")